In [2]:
import numpy as np
import pandas as pd

In [3]:
class MLP:
    def __init__(self, n_hidden, n_output=1, lr=0.1, activation=np.arctan):
        self.lr = lr
        self.batch_size = 32
        self.activation = activation
        self.hidden_nodes = n_hidden
        self.output_nodes = n_output
        
        self.bias_h = np.random.uniform(-1, 1, (self.hidden_nodes, 1))
        self.bias_o = np.random.uniform(-1, 1, (self.output_nodes, 1))
        self.weights_ih = None
        self.weights_ho = None

    def sigmoid(self, x):
        return 1/(1 + np.exp(-x))

    def dsigmoid(self, x):
        return x * (1 - x)

    def darctan(self, x):
        return 1 / (1 + np.power(x, 2))
    
    def feedforward(self, inputs):
        inputs = np.array(inputs)
        self.hidden_sum = np.matmul(self.weights_ih, np.transpose(inputs)) + self.bias_h
        
        if self.activation == np.arctan:
            self.hidden_out = self.activation(self.hidden_sum)
        else:
            self.hidden_out = self.sigmoid(self.hidden_sum)
        self.output_sum = np.matmul(self.weights_ho, self.hidden_out) + self.bias_o
        self.output_out = self.sigmoid(self.output_sum)
        return np.squeeze(self.output_out)

    
    def backprop(self, X, y):
        y = np.array(y)
        X = np.array(X)
        self.feedforward(X)
        output_deltas = y - self.output_out
        hidden_errors = np.matmul(self.weights_ho.T, output_deltas)
        if self.activation == np.arctan:
            hidden_deltas = hidden_errors * self.darctan(self.hidden_sum)
        else:
            hidden_deltas = hidden_errors * self.dsigmoid(self.hidden_out)

        self.weights_ho += self.lr * np.matmul(output_deltas, self.hidden_out.T)
        self.weights_ih += self.lr * np.matmul(hidden_deltas, X)
        self.bias_h += self.lr * np.sum(hidden_deltas, axis=1, keepdims=True)
        self.bias_o += self.lr * np.sum(output_deltas, axis=1, keepdims=True)


    def fit(self, X, y, epochs=10):
        X = np.array(X)
        y = np.array(y)
        if self.weights_ih is None:
            self.input_nodes = X.shape[1]#len(X[0])
            limit = np.sqrt(6 / (self.input_nodes + self.output_nodes))
            self.weights_ih = np.random.uniform(-limit, limit, (self.hidden_nodes, self.input_nodes))
            self.weights_ho = np.random.uniform(-limit, limit, (self.output_nodes, self.hidden_nodes))

        for epoch in range(epochs):
            indices = np.arange(X.shape[0])
            np.random.shuffle(indices)
            X = X[indices]
            y = y[indices]

            for i in range(0, X.shape[0], self.batch_size):
                X_batch = X[i:i+self.batch_size]
                y_batch = y[i:i+self.batch_size]
                self.backprop(X_batch, y_batch)


    def predict_proba(self, X):
        X = np.array(X)
        if self.weights_ih is None:
            print("Train the model first.")
            return
        preds = []
        for idx in range(len(X)):
            XX = np.array([X[idx]])
            pred = self.feedforward(XX)
            preds.append(pred)
        return np.squeeze(np.array(preds))

    def predict(self, X):
        probas = self.predict_proba(X)
        return (probas >= 0.5).astype(int)

<h3>That's just an example model, I didn't train/test it here.</h3>